# Week 44, Day 1: Hash functions

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w044d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def bad_hash(word):
    return sum(ord(ch) for ch in word)

for words in [["listen", "silent", "enlist"], ["stop", "pots", "tops", "spot"]]:
    print([bad_hash(w) for w in words], [bad_hash(w) % 8 for w in words])

In [ ]:
def slots_used(keys, m, hash_fn):
    counts = [0] * m
    for k in keys:
        counts[hash_fn(k) % m] += 1
    return counts

keys = [f"user{i}" for i in range(40)]
print(slots_used(keys, 8, lambda k: len(k)))                       # terrible: most keys have the same length
print(slots_used(keys, 8, lambda k: sum(ord(c) for c in k)))      # better, but bunched
print(slots_used(keys, 8, lambda k: int(k[4:])))                  # good for these keys only

In [ ]:
def poly_hash(text, base=31, mod=2 ** 32):
    h = 0
    for ch in text:
        h = (h * base + ord(ch)) % mod
    return h

for w in ["listen", "silent", "enlist", "cat", "act"]:
    print(f"{w:<7} {poly_hash(w):>11}  slot {poly_hash(w) % 8}")

In [ ]:
def chance_of_shared(n, days=365):
    """The probability that at least two of n people share a birthday."""
    all_different = 1.0
    for i in range(n):
        all_different *= (days - i) / days
    return 1 - all_different

for n in [10, 23, 40, 70]:
    print(n, round(chance_of_shared(n), 3))

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __eq__(self, other):
        return isinstance(other, Point) and (self.x, self.y) == (other.x, other.y)

    def __hash__(self):
        return hash((self.x, self.y))

places = {Point(1, 2): "home"}
p = Point(1, 2)
print(places[p], p in places)
p_key = next(iter(places))
p_key.x = 99                        # changing a key after it's stored...
print(Point(1, 2) in places, Point(99, 2) in places)    # ...loses it

In [ ]:
def rabin_karp(text, pattern, base=256, mod=1_000_000_007):
    k = len(pattern)
    if k == 0 or k > len(text):
        return []
    high = pow(base, k - 1, mod)                     # b^(k-1), for removing the outgoing character
    target = window = 0
    for i in range(k):
        target = (target * base + ord(pattern[i])) % mod
        window = (window * base + ord(text[i])) % mod
    found = []
    for i in range(len(text) - k + 1):
        if window == target and text[i:i + k] == pattern:     # check: hashes can collide
            found.append(i)
        if i + k < len(text):
            window = ((window - ord(text[i]) * high) * base + ord(text[i + k])) % mod
    return found

print(rabin_karp("abracadabra", "abra"), rabin_karp("aaaa", "aa"), rabin_karp("hello", "xyz"))

> **Key idea.** A hash table turns a key into a slot with `hash(key) % m`. A good hash function is deterministic, gives equal keys equal hashes, spreads similar keys far apart, and is fast. Hash strings polynomially (multiply by a base and add each character), so order matters; sums of codes make anagrams collide. Collisions are unavoidable (pigeonholes) and come early (the birthday problem), so tables must handle them. Python's `hash()` keeps equal objects equal, refuses mutable built-ins, and randomises string hashes each run against hash flooding. Keys must never change. A rolling hash updates in O(1) per step: Rabin and Karp's text search.

### Exercise: How evenly?

Write `bucket_counts(keys, m, hash_fn)`: a list of `m` counts, how many keys land in each slot `hash_fn(key) % m`. Then `worst_bucket(keys, m, hash_fn)`: the largest count. Then set `by_length`, `by_sum` and `by_poly` to the worst bucket for the 200 keys `"item0"` to `"item199"` with 16 slots, hashing by `len`, by the sum of character codes, and by the polynomial hash given in the starter.

In [ ]:
def poly_hash(text, base=31, mod=2 ** 32):
    h = 0
    for ch in text:
        h = (h * base + ord(ch)) % mod
    return h

def bucket_counts(keys, m, hash_fn):
    ___

def worst_bucket(keys, m, hash_fn):
    ___

KEYS = [f"item{i}" for i in range(200)]
by_length = ___
by_sum = ___
by_poly = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_buckets():\n    \"spreading\"\n    assert bucket_counts([\"a\", \"bb\", \"cc\", \"ddd\"], 4, len) == [0, 1, 2, 1] and sum(bucket_counts(KEYS, 16, poly_hash)) == 200\n    assert by_length == worst_bucket(KEYS, 16, len) == 100\n    assert by_sum == worst_bucket(KEYS, 16, lambda k: sum(map(ord, k))) and by_poly == worst_bucket(KEYS, 16, poly_hash)\n    assert by_poly < by_sum < by_length")

### Exercise: A polynomial hash

Write `poly_hash(text, base=31, mod=2 ** 61 - 1)` with Horner's rule (multiply, add the character code, take the remainder, for each character). Then `collisions(words, hash_fn)`: how many **pairs** of different words in the list have equal hashes.

In [ ]:
def poly_hash(text, base=31, mod=2 ** 61 - 1):
    ___

def collisions(words, hash_fn):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_poly():\n    \"hashing\"\n    assert poly_hash(\"\") == 0 and poly_hash(\"a\") == 97 and poly_hash(\"ab\") == 97 * 31 + 98 and poly_hash(\"cat\") != poly_hash(\"act\")\n    assert poly_hash(\"abc\", base=10, mod=1000) == (97 * 100 + 98 * 10 + 99) % 1000\n    words = [\"listen\", \"silent\", \"enlist\", \"stop\", \"pots\", \"tops\"]\n    assert collisions(words, lambda w: sum(map(ord, w))) == 6 and collisions(words, poly_hash) == 0\n    assert collisions([\"a\", \"a\", \"b\"], len) == 3 and collisions([], len) == 0")

### Exercise: The birthday problem

Write `chance_shared(n, slots)`: the probability that, when `n` keys are put into `slots` slots at random, at least two share a slot. Then `keys_for_half(slots)`: the smallest `n` for which that chance is at least 0.5. (For 365 slots it's 23.)

In [ ]:
def chance_shared(n, slots):
    ___

def keys_for_half(slots):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_birthday():\n    \"birthdays\"\n    assert abs(chance_shared(23, 365) - 0.5073) < 0.001 and chance_shared(1, 365) == 0 and chance_shared(366, 365) == 1\n    assert keys_for_half(365) == 23 and keys_for_half(1) == 2 and keys_for_half(1_000_000) == 1178")

### Exercise: Rabin and Karp

Write `find_all(text, pattern)`: every position where `pattern` occurs in `text` (overlapping allowed), using a rolling polynomial hash (base 256, modulus 1,000,000,007) and checking the characters only when the hashes match. An empty pattern, or one longer than the text, gives `[]`. Don't use `str.find`, `in` on strings, or regular expressions. The checks search 200,000 characters.

In [ ]:
def find_all(text, pattern):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import ast\nimport random\ndef test_small():\n    \"small\"\n    assert find_all(\"abracadabra\", \"abra\") == [0, 7] and find_all(\"aaaa\", \"aa\") == [0, 1, 2] and find_all(\"hello\", \"xyz\") == [] and find_all(\"ab\", \"abc\") == [] and find_all(\"x\", \"\") == []\n    rng = random.Random(1)\n    for _ in range(200):\n        t = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(0, 15)))\n        p = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(1, 3)))\n        assert find_all(t, p) == [i for i in range(len(t) - len(p) + 1) if t[i:i + len(p)] == p], (t, p)\ndef test_big():\n    \"200,000 characters\"\n    rng = random.Random(2)\n    t = \"\".join(rng.choice(\"acgt\") for _ in range(200_000))\n    p = t[150_000:150_040]\n    assert 150_000 in find_all(t, p)\n    tree = ast.parse(source)\n    assert not any(isinstance(n, ast.Attribute) and n.attr in (\"find\", \"index\", \"count\") for n in ast.walk(tree)) and \"import re\" not in source")

### Exercise: Hashing your own class

Write `Colour(r, g, b)` (whole numbers from 0 to 255, else `ValueError`) whose instances can be dictionary keys: `__eq__` (equal if the three numbers match; `NotImplemented` for other types), `__hash__` consistent with it, and `__repr__` like `Colour(255, 0, 0)`. Make the attributes **read-only properties**, so a key can't be changed after it's stored.

In [ ]:
class Colour:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_colour():\n    \"keys\"\n    names = {Colour(255, 0, 0): \"red\", Colour(0, 0, 255): \"blue\"}\n    assert names[Colour(255, 0, 0)] == \"red\" and len({Colour(1, 2, 3), Colour(1, 2, 3), Colour(3, 2, 1)}) == 2\n    assert repr(Colour(255, 0, 0)) == \"Colour(255, 0, 0)\" and (Colour(1, 2, 3) == (1, 2, 3)) is False\n    c = Colour(1, 2, 3)\n    for attr in \"rgb\":\n        try:\n            setattr(c, attr, 9)\n            assert False, f\"{attr} should be read-only\"\n        except AttributeError:\n            pass\n    for bad in [(256, 0, 0), (0, -1, 0)]:\n        try:\n            Colour(*bad)\n            assert False\n        except ValueError:\n            pass")

### Bug hunt: Equal but differently hashed

`Ticket` defines equality on its code, but its `__hash__` uses `id(self)`, so two equal tickets usually get different hashes, and a set holds both: this prints `2`. Hash consistently with equality, so it prints `1`.

In [ ]:
class Ticket:
    def __init__(self, code):
        self.code = code

    def __eq__(self, other):
        return isinstance(other, Ticket) and self.code == other.code

    def __hash__(self):
        return id(self)

print(len({Ticket("A7"), Ticket("A7")}))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"equal means equal hashes\"\n    expect_output(\"1\")")

### Bug hunt: Order doesn't matter, but it should

This hash multiplies the characters together, so the order of the letters doesn't count and `"ab"` and `"ba"` collide: this prints `True`. Rewrite `my_hash` as a polynomial hash (base 31, modulus `2 ** 32`), so this prints `False`.

In [ ]:
def my_hash(text):
    h = 1
    for ch in text:
        h = h * ord(ch) % 2 ** 32
    return h

print(my_hash("ab") == my_hash("ba"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"order counts\"\n    expect_output(\"False\")\n    assert my_hash(\"abc\") == ((97 * 31 + 98) * 31 + 99) % 2 ** 32")

### Bug hunt: A window that remembers too much

This rolling hash forgets to remove the outgoing character's contribution (`ord(text[i]) * high`) before shifting, so after the first window its hash is wrong and it misses the match at 4: it prints `[0]`. Fix the update, so it prints `[0, 4]`.

In [ ]:
def find_all(text, pattern, base=256, mod=1_000_000_007):
    k = len(pattern)
    high = pow(base, k - 1, mod)
    target = window = 0
    for i in range(k):
        target = (target * base + ord(pattern[i])) % mod
        window = (window * base + ord(text[i])) % mod
    found = []
    for i in range(len(text) - k + 1):
        if window == target and text[i:i + k] == pattern:
            found.append(i)
        if i + k < len(text):
            window = (window * base + ord(text[i + k])) % mod
    return found

print(find_all("abcxabc", "abc"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"remove the outgoing character\"\n    expect_output(\"[0, 4]\")")

### Puzzle: Horner's rule

Arrange the lines into a polynomial string hash. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
a = hash(7)
b = hash(-1)
c = hash(1) == hash(1.0) == hash(True)
d = hash((1, 2)) == hash((1, 2))

p = ___   # a
q = ___   # b
r = ___   # c
s = ___   # d
t = ___   # True or False: hash("cat") is the same number every time Python starts

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 7 and q == -2 and r is True, \"p, q and r\"\n    assert s is True and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: repeated stretches of DNA

A DNA string is made of `A`, `C`, `G` and `T`. Write `repeated(dna, k)`: every stretch of length `k` that appears **more than once** (overlaps allowed), in the order of its **first** appearance. Use a rolling hash so each step is O(1), with a dictionary from hash to the start positions seen; when a hash repeats, compare the actual stretches to rule out collisions (don't count a collision as a repeat). Don't build a set or dictionary of the stretches themselves. The checks use 200,000 letters and `k = 12`.

In [ ]:
def repeated(dna, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef slow(dna, k):\n    seen, out = {}, []\n    for i in range(len(dna) - k + 1):\n        s = dna[i:i + k]\n        seen[s] = seen.get(s, 0) + 1\n        if seen[s] == 2:\n            out.append(s)\n    return sorted(set(out), key=dna.find) if out else []\ndef test_small():\n    \"small\"\n    assert repeated(\"AAAAACCCCCAAAAACCCCCCAAAAAGGGTTT\", 10) == [\"AAAAACCCCC\", \"CCCCCAAAAA\"] and repeated(\"AAAAAAAAAAAAA\", 10) == [\"AAAAAAAAAA\"]\n    assert repeated(\"ACGT\", 5) == [] and repeated(\"ACGTACGT\", 4) == [\"ACGT\"] and repeated(\"ACGTACGTA\", 4) == [\"ACGT\", \"CGTA\"]\n    rng = random.Random(3)\n    for _ in range(150):\n        dna = \"\".join(rng.choice(\"AC\") for _ in range(rng.randint(0, 20)))\n        k = rng.randint(1, 5)\n        assert repeated(dna, k) == slow(dna, k), (dna, k)\ndef test_big():\n    \"200,000 letters\"\n    rng = random.Random(4)\n    dna = \"\".join(rng.choice(\"ACGT\") for _ in range(200_000))\n    got = repeated(dna, 12)\n    assert got and all(dna.count(s) >= 2 for s in got[:5])")

## Warm-up

Adding up the letter codes gives every anagram the **same** number, so they'd all go into the same slot of a table. A useful hash function has to notice **order**, not just which letters are there. That's the first lesson of today: a hash function is easy to write and easy to write badly.

## From keys to slots

A hash table is an array of `m` slots. To store or find a key:

1. compute `hash(key)`, a whole number;
2. reduce it to a slot index: `hash(key) % m`;
3. look in that slot.

If every key went to a different slot, every lookup would be one step, O(1). Different keys going to the **same** slot is called a **collision**, and the next two days are about handling them. But first, the function itself. A good hash function is:

| Property | Why |
|---|---|
| **deterministic**: the same key always gives the same hash | or you'd look in the wrong slot |
| **consistent with equality**: equal keys give equal hashes | `1`, `1.0` and `True` are equal in Python, so they must hash alike |
| **spread out**: similar keys give very different hashes | real keys are similar (`user1`, `user2`, ...), and they mustn't bunch up |
| **fast** | it runs on every single lookup |

## Hashing strings

The standard way to hash a string treats its characters as the digits of a number in some base `b`, so that order matters:

```text
hash("cat") = ord("c") × b² + ord("a") × b + ord("t")
```

Computed left to right (Horner's rule, which needs no powers): start at 0, and for each character, multiply by `b` and add its code. To keep the numbers small, take the remainder by some large `M` at each step. Java's strings use `b = 31`; a prime base helps spread the values.

Now the anagrams are far apart. The choice of **table size** matters too: if `m` is a power of two, `h % m` keeps only the last few bits of `h`, so a hash function whose low bits are poor (say, always even) would use only half the slots. Using a prime `m`, or a hash function that mixes all its bits well, avoids that.

## Collisions are certain

No hash function can avoid collisions, for two reasons:

- **The pigeonhole principle.** There are far more possible keys (every string!) than slots. Put more than `m` keys into `m` slots, and some slot must hold two.
- **The birthday problem.** Collisions start much sooner than you'd expect. In a room of just 23 people, it's more likely than not that two share a birthday, even though there are 365 days. In general, with `m` slots and keys spread at random, you expect the first collision after about `√(πm/2)` keys: for a million slots, around 1,250 keys.

So hash tables don't try to avoid collisions; they **handle** them, and keep them rare enough that the average slot holds only a few keys. Tomorrow and the day after are the two classic ways.

## Python's hash()

Python's built-in `hash()` follows rules you can rely on:

| Rule | Example |
|---|---|
| equal objects have equal hashes | `hash(1) == hash(1.0) == hash(True)` |
| ints hash to themselves (mostly) | `hash(42) == 42`; but `hash(-1) == -2`, because -1 is used inside CPython to signal an error |
| tuples hash their contents | `hash((1, 2)) == hash((1, 2))` |
| mutable built-ins can't be hashed | `hash([1, 2])` raises `TypeError` |
| **strings are hashed differently every run** | `hash("cat")` changes each time Python starts |

The last rule is a security measure. If an attacker knows the hash function, they can send a website thousands of keys that all land in the same slot, turning every lookup into a slow search and stalling the server: a **hash flooding** attack (described by Scott Crosby and Dan Wallach in 2003). So Python mixes a random secret into string hashes each time it starts (the `PYTHONHASHSEED` environment variable can fix it, for testing). That's why this lesson never prints `hash("...")`: the number would differ for you.

Why can't a list be a key? If you put a list in a set and then changed it, its hash would change, and the set would look for it in the wrong slot: it would be lost inside the set. Week 29's rule for your own classes is the same: only define `__hash__` from things that never change.

The key now says `(99, 2)` but sits in the slot for `(1, 2)`: a search for either looks in the wrong place, or finds a key that no longer matches. Keys must not change.

## Rolling hashes

A polynomial hash has a useful property: the hash of the **next** window of a string can be worked out from the previous one in O(1), by removing the outgoing character's contribution and adding the incoming one, like Week 41's sliding window total:

```text
hash(s[i+1 : i+1+k]) = (hash(s[i : i+k]) − ord(s[i]) × b^(k−1)) × b + ord(s[i+k])
```

Michael Rabin and Richard Karp (1987) used this to search for a pattern in a text: compare the **hash** of each window with the pattern's hash, and only when they match, compare the actual characters (a match of hashes might be a collision). That makes the search O(n + k) on average, instead of O(n × k) for comparing every window character by character.

`pow(base, k - 1, mod)` computes the power with the remainder taken as it goes, so the numbers stay small. Python's `%` keeps the result non-negative even after the subtraction (Week 41).

## Practice

## Quiz

## Daily challenge

## Before you go

Write down, in your own words, why a set can find an item in O(1) but a list can't, and what would happen to a set if every item had the same hash. Tomorrow: building a hash map with chaining, where each slot holds a short list of the keys that landed there.